In [1]:
import pandas as pd

In [2]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')

In [3]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [6]:
production_df['archetypes'].unique()

array(['Co hydrometallurgical refinery', 'Zn refinery',
       'Cu-Au Porphyry sulfide, flotation-based', 'Free-miling',
       'Cu-Mo Porphyry sulfide, flotation-based', nan,
       'Hybrid free-miling-refactory',
       'Ni-Cu Porphyry sulfide, flotation-based', 'Fe concentrator',
       'Ni hydrometallurgical refinery', 'Fe concentrator + pellet plant',
       'Direct Shipping Ore', 'Ni-Cu smelter',
       'Cu-Ni smelter and refinery', 'Porphyry sulfide, flotation-based',
       'Cu-Zn Porphyry sulfide, flotation-based',
       'Magnetite concentrator', 'High-grade U acid-leach mills ',
       'High-grade polymetallic Pb-Zn-Ag'], dtype=object)

In [4]:
from utils.data_manipulations import build_activity_name, add_site_id

In [5]:
# Keep only relevant columns
energy_df = energy_df[['main_id', 'facility_group_id', 'flow_type', 'subflow_type', 'value_MJ']]
material_df = material_df[['main_id', 'facility_group_id', 'flow_type', 'subflow_type', 'mass_t']]
biosphere_df = biosphere_df[['main_id', 'facility_group_id', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']]

In [6]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [7]:
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)

In [8]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')

In [9]:
# Replace column name mass_t to mass for normalization function
material_df = material_df.rename(columns={'mass_t': 'mass'})

# Data-gap filling

In [21]:
production_df

,main_id,facility_group_id,facility_name,facility_group_name,province,facility_type,mining_processing_type,archetypes,biosphere_data?,technosphere_data?,...,Mo_conc,Zn_conc,Pb_conc,Fe_conc,Pt_conc,Pd_conc,U_conc,Nb_conc,activity_name,site_id
0,QC-MAIN-089f3c60,<NA>,Bloom Lake,NaN,Quebec,mining,Open-pit,Magnetite concentrator,NPRI+GHG+Land,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,None,QC-MAIN-089f3c60
1,BC-MAIN-857b7b89,<NA>,Brucejack,NaN,British Columbia,mining,"Underground, concentrator",NaN,NPRI+GHG+Water,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Br...",BC-MAIN-857b7b89
2,QC-MAIN-de3d8b7b,<NA>,Canadian Electrolytic Zinc Limited (CEZinc),NaN,Quebec,manufacturing,Refinery,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Zn and Pb, refining at Canadian Electrolytic Z...",QC-MAIN-de3d8b7b
3,QC-MAIN-e7e6a960,<NA>,Canadian Malartic,NaN,Quebec,mining,"Open-pit, concentrator",Hybrid free-miling-refactory,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au and Ag, Open-pit mining and beneficiation a...",QC-MAIN-e7e6a960
4,NL-MAIN-dd723db4,<NA>,Carol Lake,NaN,Newfoundland and Labrador,mining,"Open-pit, concentrator",Fe concentrator + pellet plant,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,None,NL-MAIN-dd723db4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,<NA>,GRP-0a2c0d69,NaN,Meadowbank complex,Nunavut,mining,"Open-pit, underground",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au and Ag, Open-pit and underground mining at ...",GRP-0a2c0d69
62,<NA>,GRP-0d911886,NaN,Porcupine complex,Ontario,mining,"Open-pit, underground",Free-miling,NPRI+GHG+Land,Energy and materials,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Open-pit and underground mining at Porcupi...",GRP-0d911886
63,<NA>,GRP-147b3123,NaN,Timmins Operation,Ontario,mining,"Underground, concentrator",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Ti...",GRP-147b3123
64,<NA>,GRP-14bfbb82,NaN,Seabee Gold Operation,Saskatchewan,mining,"Underground, concentrator",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Se...",GRP-14bfbb82


## Normalize flows

In [10]:
from core.normalization import normalize_flows

### Per ore processed

In [11]:
energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value_MJ')

In [12]:
material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='mass')

In [13]:
biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

### Per concentrate stream

In [14]:
energy_conc_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value_MJ')

In [15]:
material_conc_econ = normalize_flows(material_df, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='mass')

In [16]:
biosphere_conc_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

### Per metal produced

In [17]:
energy_metal_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='metal', allocation='economic', value_col='value_MJ')

In [18]:
material_metal_econ = normalize_flows(material_df, production_df, price_df=price_df,  mode='metal', allocation='economic', value_col='mass')

In [19]:
biosphere_metal_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='metal', allocation='economic', value_col='value')

# Exports normalized dataframes

In [20]:
energy_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/energy_df.csv', index=False)
material_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/material_df.csv', index=False)
biosphere_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/biosphere_df.csv', index=False)

In [21]:
energy_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv', index=False)
material_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv', index=False)
biosphere_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv', index=False)

In [22]:
energy_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/energy_df.csv', index=False)
material_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/material_df.csv', index=False)
biosphere_metal_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/metal/biosphere_df.csv', index=False)